# TF-IDF From Scratch

Goal:

Convert text into numerical vectors without using TfidfVectorizer.

Concepts:

- Term Frequency (TF)
- Document Frequency (DF)
- Inverse Document Frequency (IDF)
- TF-IDF

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/emotion_sentiment_fusion_detector"

df = pd.read_csv(
    f"{PROJECT_ROOT}/data/imdb/imdb_processed.csv"
)

df.head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [4]:
subset = df.head(5000).copy()

print(subset.shape)

(5000, 3)


In [5]:
import re
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

stopwords = {
    "the","a","an","is","are","was","were",
    "be","been","being","of","to","in",
    "on","for","with","that","this","it",
    "as","at","by","from","or","and","but"
}

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"<.*?>"," ",text)

    text = re.sub(r"[^a-z\s]"," ",text)

    text = re.sub(r"\s+"," ",text)

    return text.strip()

def preprocess(text):

    text = clean_text(text)

    tokens = text.split()

    tokens = [
        word
        for word in tokens
        if word not in stopwords
    ]

    tokens = [
        stemmer.stem(word)
        for word in tokens
    ]

    return tokens

In [6]:
subset["tokens"] = subset["review"].apply(
    preprocess
)

In [7]:
vocab = set()

for tokens in subset["tokens"]:

    vocab.update(tokens)

vocab = sorted(list(vocab))

print("Vocabulary size:", len(vocab))

Vocabulary size: 26181


In [8]:
word_to_index = {
    word: idx
    for idx, word in enumerate(vocab)
}

## Term Frequency

Measures how often a word appears inside a document.

In [9]:
from collections import Counter

sample_tokens = subset.iloc[0]["tokens"]

counter = Counter(sample_tokens)

counter.most_common(10)

[('oz', 6),
 ('i', 6),
 ('me', 4),
 ('violenc', 4),
 ('show', 4),
 ('watch', 3),
 ('you', 3),
 ('ll', 3),
 ('not', 3),
 ('prison', 3)]

In [10]:
def compute_tf(tokens):

    counts = Counter(tokens)

    total_words = len(tokens)

    tf = {}

    for word,count in counts.items():

        tf[word] = count / total_words

    return tf

In [11]:
sample_tf = compute_tf(
    sample_tokens
)

list(sample_tf.items())[:10]

[('one', 0.0045871559633027525),
 ('other', 0.009174311926605505),
 ('review', 0.0045871559633027525),
 ('ha', 0.0045871559633027525),
 ('mention', 0.0045871559633027525),
 ('after', 0.0045871559633027525),
 ('watch', 0.013761467889908258),
 ('just', 0.009174311926605505),
 ('oz', 0.027522935779816515),
 ('episod', 0.009174311926605505)]

## Inverse Document Frequency

Words appearing in many documents receive lower importance.

In [12]:
from collections import defaultdict

df_counts = defaultdict(int)

for tokens in subset["tokens"]:

    unique_words = set(tokens)

    for word in unique_words:

        df_counts[word] += 1

In [13]:
import math

num_docs = len(subset)

idf = {}

for word,count in df_counts.items():

    idf[word] = math.log(
        num_docs / (count + 1)
    )

In [17]:
idf["movi"]

0.4228147464432773

In [16]:
idf["masterpiec"]

3.6651629274966204

## TF-IDF

TF-IDF = TF × IDF

In [18]:
def compute_tfidf(tokens):

    tf = compute_tf(tokens)

    tfidf = {}

    for word,value in tf.items():

        tfidf[word] = value * idf[word]

    return tfidf

In [19]:
sample_tfidf = compute_tfidf(
    sample_tokens
)

sorted(
    sample_tfidf.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]

[('oz', 0.1469487897541732),
 ('inmat', 0.0617012268090659),
 ('violenc', 0.06009992984140542),
 ('prison', 0.056234537512471434),
 ('forget', 0.04686062481274809),
 ('struck', 0.04663491731129442),
 ('penitentari', 0.03589011931585455),
 ('unflinch', 0.03403018762728499),
 ('inward', 0.032710545093102505),
 ('privaci', 0.032710545093102505),
 ('nickel', 0.032710545093102505),
 ('timid', 0.03168695082101898),
 ('emerald', 0.03168695082101898),
 ('gangsta', 0.03168695082101898),
 ('scuffl', 0.03168695082101898),
 ('ll', 0.031120777288938584),
 ('oswald', 0.03085061340453295),
 ('aryan', 0.03085061340453295),
 ('shadi', 0.03085061340453295),
 ('due', 0.030600088450680296)]

In [20]:
top_words = sorted(
    df_counts.items(),
    key=lambda x:x[1],
    reverse=True
)[:3000]

vocab = [
    word
    for word,_ in top_words
]

In [21]:
word_to_index = {
    word:i
    for i,word in enumerate(vocab)
}

In [22]:
import numpy as np

def vectorize(tokens):

    vector = np.zeros(len(vocab))

    tfidf = compute_tfidf(tokens)

    for word,value in tfidf.items():

        if word in word_to_index:

            idx = word_to_index[word]

            vector[idx] = value

    return vector

In [23]:
X = np.array([
    vectorize(tokens)
    for tokens in subset["tokens"]
])

In [24]:
y = subset["label"].values

In [25]:
print(X.shape)
print(y.shape)

(5000, 3000)
(5000,)


In [26]:
import os
import numpy as np

os.makedirs(
    f"{PROJECT_ROOT}/data/tfidf",
    exist_ok=True
)

In [27]:
np.save(
    f"{PROJECT_ROOT}/data/tfidf/X.npy",
    X
)

np.save(
    f"{PROJECT_ROOT}/data/tfidf/y.npy",
    y
)

print("Saved")

Saved


# TF-IDF Summary

Implemented:

- Vocabulary Construction
- Term Frequency
- Document Frequency
- Inverse Document Frequency
- TF-IDF Weighting
- Vector Representation

The text dataset has now been converted into numerical features suitable for machine learning.